# Part 1 - Custom CNNs on STL-10

**Authors**: Azaliia Agisheva, Jose Manuel Bueno

This notebook implements **several** custom CNN pipelines for STL-10 using TensorFlow Datasets:
- data loading from `tfds`
- preprocessing (resize, normalization, one-hot labels)
- train/validation split
- **four** from-scratch architectures for comparison: **baseline** (stacked convs), **residual** (skip connections), **Inception-style** (parallel branches), and **Xception-inspired** (depthwise separable convolutions + residual shortcuts, following lab U3.3 §3.11)
- shared training setup (run training / eval cells when ready)
- a short **results comparison** section to fill after training


## 1) Imports and global configuration

We set seeds for reproducibility and define core hyperparameters.

In [72]:
import os
import random
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, regularizers

SEED = 57
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE
IMG_SIZE = (96, 96)
NUM_CLASSES = 10
BATCH_SIZE = 64

# STL-10 labeled train: 5,000 images; we use 80% for training (~4,000) for this split.
TRAIN_SAMPLES = 4000
STEPS_PER_EPOCH = (TRAIN_SAMPLES + BATCH_SIZE - 1) // BATCH_SIZE

# Training / regularization (tuned to reduce overfitting on small STL-10 train)
LABEL_SMOOTHING = 0.05
USE_MIXUP = True
MIXUP_ALPHA = 0.2

# STL-10 has 5,000 labeled train images and 8,000 test images.
TRAIN_SPLIT = "train[:80%]"
VAL_SPLIT = "train[80%:]"
TEST_SPLIT = "test"


## 2) Load STL-10 from TensorFlow Datasets

In [73]:
# as_supervised=True -> each item is (image, label)
train_raw, val_raw, test_raw = tfds.load(
    "stl10",
    split=[TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT],
    as_supervised=True,
    shuffle_files=True,
)

builder = tfds.builder("stl10")
class_names = builder.info.features["label"].names
print("Classes:", class_names)
print("Num classes:", len(class_names))

Classes: ['airplane', 'bird', 'car', 'cat', 'deer', 'dog', 'horse', 'monkey', 'ship', 'truck']
Num classes: 10


## 3) Preprocessing and `tf.data` pipelines

Best practices used:
- resize images to fixed shape
- normalize pixels to `[0, 1]`
- one-hot encode labels
- shuffle only train split
- cache + prefetch for throughput

In [74]:
def preprocess_example(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    label = tf.one_hot(label, depth=NUM_CLASSES)
    return image, label


def mixup_batch(images, labels):
    """Mixup (Zhang et al.) on batched tensors; labels are one-hot."""
    alpha = tf.cast(MIXUP_ALPHA, tf.float32)
    batch_size = tf.shape(images)[0]
    # Beta(a,b) = G1/(G1+G2) with G1,G2 ~ Gamma(shape); avoids tf.random.beta (not in all TF builds)
    shape_1d = tf.stack([batch_size])
    g1 = tf.random.gamma(shape_1d, alpha, dtype=tf.float32)
    g2 = tf.random.gamma(shape_1d, alpha, dtype=tf.float32)
    lam = g1 / (g1 + g2 + 1e-8)
    lam = tf.cast(lam, dtype=tf.float32)
    lam_x = tf.reshape(lam, [-1, 1, 1, 1])
    lam_y = tf.reshape(lam, [-1, 1])
    order = tf.random.shuffle(tf.range(batch_size, dtype=tf.int32))
    images_2 = tf.gather(images, order)
    labels_2 = tf.gather(labels, order)
    mixed_images = lam_x * images + (1.0 - lam_x) * images_2
    mixed_labels = lam_y * labels + (1.0 - lam_y) * labels_2
    return mixed_images, mixed_labels


train_ds = (
    train_raw.map(preprocess_example, num_parallel_calls=AUTOTUNE)
    .shuffle(5000, seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
)
if USE_MIXUP:
    train_ds = train_ds.map(mixup_batch, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.prefetch(AUTOTUNE)

val_ds = (
    val_raw.map(preprocess_example, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    test_raw.map(preprocess_example, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Train/Val/Test pipelines ready.", "Mixup on train:" if USE_MIXUP else "Mixup off.")


Train/Val/Test pipelines ready. Mixup on train:


## 4) Data augmentation

**`baseline_augmentation`** — flip, rotation, zoom, contrast (same for all models).

Optional **mixup** is applied on batched train data in the preprocessing cell.


In [75]:
baseline_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.1),
    ],
    name="baseline_augmentation",
)


## 5) Custom CNN architectures (from scratch)

We train **four** models under the same data pipeline and optimizer so results are comparable:

| Model | Idea |
|-------|------|
| **Baseline** | Stacked conv blocks (BN, dropout, pooling) — simple depth baseline |
| **Residual** | Residual blocks with skip connections — eases optimization in deeper stacks |
| **Inception-style** | Naive Inception blocks (1×1, 3×3, 5×5, pool+projection); branch width **c** (48 → 56) controls capacity vs ~82k params when **c=24** |
| **Xception-inspired** | `SeparableConv2D` stacks with residual projections — depthwise separable convs (U3.3 §3.11.4 pattern: regular `Conv2D` stem, then separable blocks + `add`) |

All models use **L2** regularization, **Batch Normalization**, **Dropout**, and **Global Average Pooling** before the classifier head. No `keras.applications` / pretrained weights.


In [76]:
def conv_bn_relu(x, filters, kernel_size=3, wd=1e-4):
    x = layers.Conv2D(
        filters,
        kernel_size,
        padding="same",
        use_bias=False,
        kernel_regularizer=regularizers.l2(wd),
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x


def conv_bn_relu_1x1(x, filters, wd=1e-4):
    return conv_bn_relu(x, filters, kernel_size=1, wd=wd)


def conv_block(x, filters, kernel_size=3, pool=True, dropout=0.2, wd=1e-4):
    x = layers.Conv2D(
        filters,
        kernel_size,
        padding="same",
        use_bias=False,
        kernel_regularizer=regularizers.l2(wd),
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(
        filters,
        kernel_size,
        padding="same",
        use_bias=False,
        kernel_regularizer=regularizers.l2(wd),
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    if pool:
        x = layers.MaxPooling2D(pool_size=2)(x)

    x = layers.Dropout(dropout)(x)
    return x


# def residual_block(x, filters, dropout, wd, project_first=True):
#     if project_first:
#         shortcut = layers.Conv2D(
#             filters,
#             1,
#             padding="same",
#             use_bias=False,
#             kernel_regularizer=regularizers.l2(wd),
#         )(x)
#         shortcut = layers.BatchNormalization()(shortcut)
#     else:
#         shortcut = x
#     y = conv_bn_relu(x, filters, wd=wd)
#     y = layers.Dropout(dropout * 0.85)(y)
#     y = conv_bn_relu(y, filters, wd=wd)
#     y = layers.Dropout(dropout)(y)
#     return layers.Add()([shortcut, y])

def residual_block(x, filters, dropout, wd, project_first=True):
    if project_first or x.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters, 1, padding="same", use_bias=False,
            kernel_regularizer=regularizers.l2(wd),
        )(x)
        shortcut = layers.BatchNormalization()(shortcut)
    else:
        shortcut = x

    y = layers.Conv2D(
        filters, 3, padding="same", use_bias=False,
        kernel_regularizer=regularizers.l2(wd),
    )(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation("relu")(y)
    y = layers.Dropout(dropout)(y)

    y = layers.Conv2D(
        filters, 3, padding="same", use_bias=False,
        kernel_regularizer=regularizers.l2(wd),
    )(y)
    y = layers.BatchNormalization()(y)

    y = layers.Add()([shortcut, y])
    y = layers.Activation("relu")(y)
    return y


def naive_inception_block(x, c=48, wd=1e-4):
    """Naive Inception: c filters per branch (4 branches -> 4c channels). Wider c = more capacity (was 24 -> ~81k params total)."""
    b1 = conv_bn_relu_1x1(x, c, wd=wd)

    b3 = conv_bn_relu_1x1(x, c, wd=wd)
    b3 = conv_bn_relu(b3, c, kernel_size=3, wd=wd)

    b5 = conv_bn_relu_1x1(x, c, wd=wd)
    b5 = conv_bn_relu(b5, c, kernel_size=5, wd=wd)

    bp = layers.MaxPooling2D(3, strides=1, padding="same")(x)
    bp = conv_bn_relu_1x1(bp, c, wd=wd)

    y = layers.Concatenate(axis=-1)([b1, b3, b5, bp])
    y = layers.BatchNormalization()(y)
    y = layers.Activation("relu")(y)
    y = layers.Dropout(0.2)(y)
    return y


def classification_head(x, num_classes, wd=1e-4, head_dropout=0.5):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(wd))(x)
    x = layers.Dropout(head_dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)


def build_model_baseline(input_shape=(96, 96, 3), num_classes=10, wd=1e-4):
    inputs = keras.Input(shape=input_shape)
    x = baseline_augmentation(inputs)
    x = conv_block(x, 32, dropout=0.15, wd=wd)
    x = conv_block(x, 64, dropout=0.20, wd=wd)
    x = conv_block(x, 128, dropout=0.25, wd=wd)
    x = conv_block(x, 256, dropout=0.30, wd=wd)
    # Original baseline head: 0.4 dropout (shared head was later set to 0.5 for other models).
    outputs = classification_head(x, num_classes, wd=wd, head_dropout=0.4)
    return keras.Model(inputs, outputs, name="baseline_stl10_cnn")


def build_model_residual(input_shape=(96, 96, 3), num_classes=10, wd=1e-4):
    """Residual CNN: stronger L2, higher dropouts, extra dropout between convs in each block."""
    inputs = keras.Input(shape=input_shape)
    x = baseline_augmentation(inputs)
    x = conv_bn_relu(x, 64, wd=wd)
    x = layers.MaxPooling2D(2)(x)
    x = residual_block(x, 64, 0.28, wd, project_first=True)
    x = residual_block(x, 64, 0.28, wd, project_first=False)
    x = layers.MaxPooling2D(2)(x)
    x = residual_block(x, 128, 0.32, wd, project_first=True)
    x = residual_block(x, 128, 0.32, wd, project_first=False)
    x = layers.MaxPooling2D(2)(x)
    x = residual_block(x, 256, 0.38, wd, project_first=True)
    outputs = classification_head(x, num_classes, wd=wd)
    return keras.Model(inputs, outputs, name="residual_stl10_cnn")


def build_model_inception(input_shape=(96, 96, 3), num_classes=10, wd=1e-4):
    inputs = keras.Input(shape=input_shape)
    x = baseline_augmentation(inputs)
    x = conv_bn_relu(x, 32, wd=wd)
    x = layers.MaxPooling2D(2)(x)
    x = naive_inception_block(x, c=48, wd=wd)
    x = layers.MaxPooling2D(2)(x)
    x = naive_inception_block(x, c=56, wd=wd)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.3)(x)
    outputs = classification_head(x, num_classes, wd=wd)
    return keras.Model(inputs, outputs, name="inception_stl10_cnn")


    

def build_model_xception(input_shape=(96, 96, 3), num_classes=10, wd=1e-4):
    """Xception-inspired depthwise separable CNN (lab U3.3 §3.11.4).

    Regular Conv2D stem first, then SeparableConv2D pairs + residual; light dropout after each add.
    """
    reg = regularizers.l2(wd)
    inputs = keras.Input(shape=input_shape)
    x = baseline_augmentation(inputs)
    x = layers.Conv2D(
        32,
        5,
        padding="same",
        use_bias=False,
        kernel_regularizer=reg,
    )(x)
    for size in [32, 64, 128, 256, 512]:
        residual = x
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(
            size,
            3,
            padding="same",
            use_bias=False,
            depthwise_regularizer=reg,
            pointwise_regularizer=reg,
        )(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(
            size,
            3,
            padding="same",
            use_bias=False,
            depthwise_regularizer=reg,
            pointwise_regularizer=reg,
        )(x)
        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)
        residual = layers.Conv2D(
            size,
            1,
            strides=2,
            padding="same",
            use_bias=False,
            kernel_regularizer=reg,
        )(residual)
        x = layers.add([x, residual])
        x = layers.Dropout(0.12)(x)
    outputs = classification_head(x, num_classes, wd=wd)
    return keras.Model(inputs, outputs, name="xception_stl10_cnn")


MODEL_BUILDERS = [
    ("baseline", build_model_baseline),
    ("residual", build_model_residual),
    ("inception", build_model_inception),
    ("xception", build_model_xception),
]

models = {}
for name, builder in MODEL_BUILDERS:
    models[name] = builder(input_shape=IMG_SIZE + (3,), num_classes=NUM_CLASSES)
    print(f"\n{'=' * 60}\n{name.upper()} — summary\n{'=' * 60}")
    models[name].summary()



BASELINE — summary


Model: "baseline_stl10_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_93 (InputLayer)     │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ baseline_augmentation           │ (None, 96, 96, 3)      │             0 │
│ (Sequential)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_847 (Conv2D)             │ (None, 96, 96, 32)     │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_956         │ (None, 96, 96, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_890 (Activation)     │ (None, 96, 96, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_848 (Conv2D)             │ (None, 96, 96, 32)     │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_957         │ (None, 96, 96, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_891 (Activation)     │ (None, 96, 96, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_323               │ (None, 48, 48, 32)     │             0 │
│ (MaxPooling2D)                  │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_446 (Dropout)           │ (None, 48, 48, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_849 (Conv2D)             │ (None, 48, 48, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_958         │ (None, 48, 48, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_892 (Activation)     │ (None, 48, 48, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_850 (Conv2D)             │ (None, 48, 48, 64)     │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_959         │ (None, 48, 48, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_893 (Activation)     │ (None, 48, 48, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_324               │ (None, 24, 24, 64)     │             0 │
│ (MaxPooling2D)                  │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_447 (Dropout)           │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_851 (Conv2D)             │ (None, 24, 24, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_960         │ (None, 24, 24, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_894 (Activation)     │ (None, 24, 24, 128)    │             

 Total params: 1,243,498 (4.74 MB)

 Trainable params: 1,241,578 (4.74 MB)

...

Model: "xception_stl10_cnn"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_97      │ (None, 96, 96, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ baseline_augmentat… │ (None, 96, 96, 3) │          0 │ input_layer_97[0… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_882 (Conv2D) │ (None, 96, 96,    │      2,400 │ baseline_augment… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 96, 96,    │        128 │ conv2d_882[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_924      │ (None, 96, 96,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_1… │ (None, 96, 96,    │      1,312 │ activation_924[0… │
│ (SeparableConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 96, 96,    │        128 │ separable_conv2d… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_925      │ (None, 96, 96,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_1… │ (None, 96, 96,    │      1,312 │ activation_925[0… │
│ (SeparableConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_335   │ (None, 48, 48,    │          0 │ separable_conv2d… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_883 (Conv2D) │ (None, 48, 48,    │      1,024 │ conv2d_882[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_195 (Add)       │ (None, 48, 48,    │          0 │ max_pooling2d_33… │
│                     │ 32)               │            │ conv2d_883[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_461         │ (None, 48, 48,    │          0 │ add_195[0][0]     │
│ (Dropout)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 48, 48,    │        128 │ dropout_461[0][0] │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_926      │ (None, 48, 48,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_1… │ (None, 48, 48,    │      2,336 │ activation_926[0… │
│ (SeparableConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 48, 48,    │        256 │ separable_conv2d

 Total params: 855,242 (3.26 MB)

 Trainable params: 852,234 (3.25 MB)

 Non-trainable params: 3,008 (11.75 KB)

## 6) Compile and training (all models)

**Baseline** and **residual**: **Adam 1e-3**, **label smoothing 0.1**, **ReduceLROnPlateau**.

**Inception** and **Xception**: **cosine decay**, **`LABEL_SMOOTHING`**.

Run this cell to train all four models (GPU recommended).


In [77]:
EPOCHS = 80

# Cosine decay for inception / xception only
total_steps = STEPS_PER_EPOCH * EPOCHS


def make_cosine_optimizer():
    lr = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=1e-3,
        decay_steps=total_steps,
        alpha=1e-4 / 1e-3,
    )
    return keras.optimizers.Adam(learning_rate=lr)


def make_callbacks(model_name: str, plateau_lr: bool):
    safe = model_name.replace(" ", "_")
    cbs = [
        keras.callbacks.ModelCheckpoint(
            filepath=f"best_{safe}_stl10.keras",
            monitor="val_accuracy",
            save_best_only=True,
            mode="max",
            verbose=1,
        ),
    ]
    if plateau_lr:
        cbs.append(
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=3,
                min_lr=1e-6,
                verbose=1,
            )
        )
    cbs.append(
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=8 if plateau_lr else 12,
            min_delta=5e-4,
            restore_best_weights=True,
            verbose=1,
        )
    )
    return cbs


results_history = {}
trained_models = {}

for name, builder in MODEL_BUILDERS:
    print(f"\n{'#' * 60}\nTraining: {name}\n{'#' * 60}")
    m = builder(input_shape=IMG_SIZE + (3,), num_classes=NUM_CLASSES)
    if name in ("baseline", "residual"):
        m.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
            metrics=["accuracy"],
        )
        callbacks = make_callbacks(name, plateau_lr=True)
    else:
        m.compile(
            optimizer=make_cosine_optimizer(),
            loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
            metrics=["accuracy"],
        )
        callbacks = make_callbacks(name, plateau_lr=False)
    history = m.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    results_history[name] = history
    trained_models[name] = m
print("Done training all models.")



############################################################
Training: baseline
############################################################
Epoch 1/80


E0000 00:00:1774393129.094582   13227 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/baseline_stl10_cnn_1/dropout_467_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


62/63 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.2072 - loss: 2.4079
Epoch 1: val_accuracy improved from None to 0.15900, saving model to best_baseline_stl10.keras
63/63 ━━━━━━━━━━━━━━━━━━━━ 9s 93ms/step - accuracy: 0.2370 - loss: 2.2615 - val_accuracy: 0.1590 - val_loss: 2.4256 - learning_rate: 0.0010
Epoch 2/80
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.2719 - loss: 2.1125
Epoch 2: val_accuracy did not improve from 0.15900
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 90ms/step - accuracy: 0.2820 - loss: 2.1044 - val_accuracy: 0.1350 - val_loss: 2.6839 - learning_rate: 0.0010
Epoch 3/80
62/63 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.3111 - loss: 2.0681
Epoch 3: val_accuracy did not improve from 0.15900
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 94ms/step - accuracy: 0.3080 - loss: 2.0655 - val_accuracy: 0.1430 - val_loss: 2.5104 - learning_rate: 0.0010
Epoch 4/80
62/63 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.3120 - loss: 2.0674
Epoch 4: val_accuracy improved from 0.15900 to 0.1870

E0000 00:00:1774393459.416594   13227 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/residual_stl10_cnn_1/dropout_472_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.2178 - loss: 2.4593
Epoch 1: val_accuracy improved from None to 0.16900, saving model to best_residual_stl10.keras
63/63 ━━━━━━━━━━━━━━━━━━━━ 15s 140ms/step - accuracy: 0.2428 - loss: 2.3271 - val_accuracy: 0.1690 - val_loss: 2.5546 - learning_rate: 0.0010
Epoch 2/80
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.3096 - loss: 2.1585
Epoch 2: val_accuracy did not improve from 0.16900
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 130ms/step - accuracy: 0.3075 - loss: 2.1527 - val_accuracy: 0.1180 - val_loss: 3.2242 - learning_rate: 0.0010
Epoch 3/80
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.3150 - loss: 2.1499
Epoch 3: val_accuracy did not improve from 0.16900
63/63 ━━━━━━━━━━━━━━━━━━━━ 9s 134ms/step - accuracy: 0.3215 - loss: 2.1390 - val_accuracy: 0.1020 - val_loss: 3.8172 - learning_rate: 0.0010
Epoch 4/80
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - accuracy: 0.3379 - loss: 2.0886
Epoch 4: val_accuracy did not improve from 0.1

E0000 00:00:1774393864.966949   13227 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/inception_stl10_cnn_1/dropout_478_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step - accuracy: 0.2244 - loss: 2.4479
Epoch 1: val_accuracy improved from None to 0.14600, saving model to best_inception_stl10.keras
63/63 ━━━━━━━━━━━━━━━━━━━━ 18s 151ms/step - accuracy: 0.2430 - loss: 2.2706 - val_accuracy: 0.1460 - val_loss: 2.3866
Epoch 2/80
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - accuracy: 0.2912 - loss: 2.0547
Epoch 2: val_accuracy did not improve from 0.14600
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 113ms/step - accuracy: 0.3115 - loss: 2.0319 - val_accuracy: 0.1030 - val_loss: 2.4372
Epoch 3/80
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.3038 - loss: 1.9948
Epoch 3: val_accuracy improved from 0.14600 to 0.14700, saving model to best_inception_stl10.keras
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 118ms/step - accuracy: 0.3200 - loss: 1.9890 - val_accuracy: 0.1470 - val_loss: 2.4538
Epoch 4/80
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - accuracy: 0.3619 - loss: 1.9552
Epoch 4: val_accuracy did not improve from 0.14700
63/63 ━━━━━━━━━━━━

E0000 00:00:1774394435.344390   13227 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/xception_stl10_cnn_1/dropout_482_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - accuracy: 0.1595 - loss: 2.5016
Epoch 1: val_accuracy improved from None to 0.14500, saving model to best_xception_stl10.keras
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 155ms/step - accuracy: 0.1848 - loss: 2.4337 - val_accuracy: 0.1450 - val_loss: 2.5314
Epoch 2/80
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.2607 - loss: 2.2469
Epoch 2: val_accuracy did not improve from 0.14500
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 119ms/step - accuracy: 0.2648 - loss: 2.2341 - val_accuracy: 0.1440 - val_loss: 2.5037
Epoch 3/80
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.2845 - loss: 2.1593
Epoch 3: val_accuracy improved from 0.14500 to 0.14600, saving model to best_xception_stl10.keras
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 118ms/step - accuracy: 0.2945 - loss: 2.1510 - val_accuracy: 0.1460 - val_loss: 2.5126
Epoch 4/80
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.3204 - loss: 2.0916
Epoch 4: val_accuracy did not improve from 0.14600
63/63 ━━━━━━━━━━━━━━

## 
7) Evaluation and comparison

Run the next cell after training to evaluate each model on the test set and print a summary table. If you skipped training in this session, load checkpoints first or re-run the training cell.


In [78]:
if not trained_models:
    print("No trained models in memory — run the training cell first (or load weights into `trained_models`).")
else:
    summary_rows = []
    for name, m in trained_models.items():
        test_loss, test_acc = m.evaluate(test_ds, verbose=1)
        val_acc = max(results_history[name].history.get("val_accuracy") or [float("nan")])
        n_params = m.count_params()
        summary_rows.append((name, val_acc, test_acc, test_loss, n_params))
        print(f"[{name}] best val acc (epoch max): {val_acc:.4f} | test acc: {test_acc:.4f}")


    print("| Model | Val acc (best epoch) | Test acc | Params |")
    print("|-------|----------------------|----------|--------|")
    for name, val_acc, test_acc, test_loss, n_params in summary_rows:
        print(f"| {name} | {val_acc:.4f} | {test_acc:.4f} | {n_params:,} |")


125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.6162 - loss: 1.4075
[baseline] best val acc (epoch max): 0.6140 | test acc: 0.6162
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.6415 - loss: 1.4358
[residual] best val acc (epoch max): 0.6270 | test acc: 0.6415
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.6291 - loss: 1.3335
[inception] best val acc (epoch max): 0.6130 | test acc: 0.6291
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7122 - loss: 1.1691
[xception] best val acc (epoch max): 0.7140 | test acc: 0.7122
| Model | Val acc (best epoch) | Test acc | Params |
|-------|----------------------|----------|--------|
| baseline | 0.6140 | 0.6162 | 1,243,498 |
| residual | 0.6270 | 0.6415 | 1,670,602 |
| inception | 0.6130 | 0.6291 | 299,434 |
| xception | 0.7140 | 0.7122 | 855,242 |


## 8) Comment results

### Mixup (`USE_MIXUP`)

**Mixup** (batched convex combinations of images and one-hot labels) acts as a **strong regularizer**. In earlier runs it **helped** residual, inception, and xception vs training without mixup, but **hurt peak baseline validation** (e.g. best val **~0.668** without mixup vs **~0.64** with mixup(in the previous run)). In the **latest** evaluation below, the **baseline** remains in the **low–mid 0.61s** on validation, while **deeper models** (especially Xception) reach **higher** val/test — consistent with mixup pairing better with **more capacity**.

### Summary table (latest run)

| Model | Val acc (best epoch) | Test acc | Params |
|-------|----------------------|----------|--------|
| baseline | 0.6140 | 0.6162 | 1,243,498 |
| residual | 0.6270 | **0.6415** | 1,670,602 |
| inception | 0.6130 | 0.6291 | 299,434 |
| xception | **0.7140** | **0.7122** | 855,242 |

### Per-model comments

**Baseline** — **Val 0.6140** and **test 0.6162** are almost identical → **stable generalization**, little sign of overfitting on the held-out test set. Accuracy is **solid** for a custom stack on STL-10 but **below** the best specialized architectures in this comparison.

**Residual** — **Best test accuracy among the three non-Xception models (0.6415)**; test is **above** validation (**0.6270**), which can happen with a **noisy 1k-sample val split** or lucky test draw. Skip connections plus the stronger L2/dropout recipe here **pay off** vs the baseline on **test** in this run.

**Inception-style** — After **widening branches** (higher **c**, **~299k** parameters vs the earlier **~82k** narrow design), **test accuracy (0.6291)** is **much closer** to baseline/residual and **no longer the weakest** model. Val (**0.6130**) and test are **aligned** — good sign that the previous gap was largely **under-capacity**, not only regularization.

**Xception-inspired** — Still the **strongest** model: **0.7140** val / **0.7122** test, with **val ≈ test** → excellent **generalization**. Depthwise-separable blocks with residual shortcuts plus this training setup give the best **accuracy / effort** trade-off here (**~855k** params).

### Note on Inception parameter count (before vs after widening)

The first **narrow** Inception (**c = 24** per branch, **~82k** params) was **parameter-efficient by design** but **capacity-limited**; widening to **c = 48 / 56** brought the model to **~299k** parameters and **clearly improved** test performance in the latest run, supporting the view that the earlier design was **too small** for STL-10 rather than mis-counted.
